<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/CNN_%7C_Chest_Xray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Libraries and Data Access**

In [ ]:
# Libraries (install torch/torchvision matching your environment if needed).
import os
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as transforms  # Image preprocessing library
from torch.optim import Adam
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Seeds improve repeatability; GPU operations are not guaranteed bitwise identical.

In [ ]:
#@title Constructing the Path
import kagglehub
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# **Data Preparation**

In [ ]:
# Accessing the chest_xray folder; avoid recursively loading nested duplicate copies.
candidates = [Path(path) / 'chest_xray', Path(path) / 'chest_xray' / 'chest_xray', Path(path)]
chest_xray_path = next((root for root in candidates
                       if all((root / split).is_dir() for split in ['train', 'val', 'test'])), None)
if chest_xray_path is None:
    raise FileNotFoundError('Expected train/val/test folders in the downloaded dataset.')
print(chest_xray_path)

In [ ]:
# Transform Function
width = height = 256
# Preserve the full field of view instead of randomly cropping away lung regions.
# Square resizing is a simple baseline and does change the original aspect ratio.
transformation = transforms.Compose([
    transforms.Resize((height, width), antialias=True),
    transforms.Grayscale(num_output_channels=1),  # Convert to 1 channel (grayscale)
    transforms.RandomRotation(5),  # Mild augmentation; inspect transformed examples
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),  # Convert pixels to [0, 1]
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
# Transformation for validation and test sets (no random augmentations).
val_test_transformation = transforms.Compose([
    transforms.Resize((height, width), antialias=True),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
# Fixed mean/std above map [0, 1] to [-1, 1]; they are not learned dataset statistics.

In [ ]:
#@title Creating ImageFolder & DataLoaders

# Define the paths for train, validation, and test directories
train_dir = os.path.join(chest_xray_path, 'train')
val_dir = os.path.join(chest_xray_path, 'val')
test_dir = os.path.join(chest_xray_path, 'test')

# Create ImageFolder datasets
train_dataset = torchvision.datasets.ImageFolder(root=train_dir, transform=transformation)
val_dataset = torchvision.datasets.ImageFolder(root=val_dir, transform=val_test_transformation)
test_dataset = torchvision.datasets.ImageFolder(root=test_dir, transform=val_test_transformation)

# Define batch sizes
batch_size = 32

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

valid_N = len(val_loader.dataset)

# Print the number of samples in each dataset
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

print(f"Classes: {train_dataset.classes}")
# Class-to-index mappings must match across all splits.
assert train_dataset.class_to_idx == val_dataset.class_to_idx == test_dataset.class_to_idx
assert train_dataset.classes == ['NORMAL', 'PNEUMONIA']
counts = np.bincount(train_dataset.targets, minlength=2)
print('Training class counts:', dict(zip(train_dataset.classes, counts.tolist())))
print(f'Majority-class training baseline: {counts.max() / counts.sum():.3f}')
if len(val_dataset) < 100:
    print('CAUTION: very small validation set; checkpoint choice will be noisy. '
          'Obtain a larger patient-disjoint validation split before extensive tuning.')

# Detect identical decoded images across splits (even if file names differ).
# This does not detect all near-duplicates or different images from one patient.
import hashlib
from PIL import Image
seen = {}
for split, dataset in [('train', train_dataset), ('validation', val_dataset), ('test', test_dataset)]:
    for filename, label in dataset.samples:
        with Image.open(filename) as image:
            image = image.convert('L')
            digest = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
        if digest in seen and seen[digest][0] != split:
            raise ValueError(f'Identical image across splits: {seen[digest][1]} and {filename}. '
                             'Resolve duplicates and patient grouping before training.')
        seen[digest] = (split, filename)
print('No identical decoded images found across the three splits.')

# **Model Creation**

In [ ]:
n_classes = 2
kernel_size = 3 # Size of the convolutional filter
img_chs = 1 # 1 because the x-ray images are grayscale
flattened_img_size = 75 * 4 * 4  # Adaptive pooling limits the dense layer size

model = nn.Sequential(
    # First convolution
    nn.Conv2d(img_chs, 25, kernel_size, stride=1, padding=1),  # Output 25 channels, spatial size 256x256
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # Downsamples the feature map to 128x128
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # Output 50 channels, spatial size 128x128
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # Output 50 channels, spatial size 64x64
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # Output 75 channels, spatial size 64x64
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # Output 75 channels, spatial size 32x32
    # Flatten to Dense
    nn.AdaptiveAvgPool2d((4, 4)),  # Compact head independent of input spatial size
    nn.Flatten(),
    nn.Linear(flattened_img_size, 128),
    nn.Dropout(.4),
    nn.ReLU(),
    nn.Linear(128, n_classes)
)

# batch normalization scales the values in the hidden layers to improve training.
# The padding helps maintain the spatial dimensions.

In [ ]:
# Move the model to the chosen device.
# torch.compile is optional and adds setup/compilation complexity for this small project.
model = model.to(device)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Criterion and Optimizer
# Keep standard cross-entropy as the starting objective; inspect per-class results.
# It expects raw logits: do not apply softmax before this loss.
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

In [ ]:
#@title Batch Accuracy
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

# **Training the Model**

In [ ]:
#@title Training Function

def train():
  total_loss = 0
  total_correct = 0
  total_samples = 0

  model.train()
  for x,y in train_loader:
      x, y = x.to(device), y.to(device) # Move data to the specified device
      output= model(x)
      optimizer.zero_grad()
      batch_loss = criterion(output, y)
      batch_loss.backward()
      optimizer.step()

      total_loss += batch_loss.item() * len(x) # Accumulate weighted loss
      pred = output.argmax(dim=1, keepdim=True)
      total_correct += pred.eq(y.view_as(pred)).sum().item()
      total_samples += len(x)

  avg_loss = total_loss / total_samples
  avg_accuracy = total_correct / total_samples
  print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(avg_loss, avg_accuracy))
  return avg_loss, avg_accuracy

In [ ]:
# Validation Function: validation loader only during model development.
# Explicit loader argument allows a separate final test pass after training.
def evaluate(loader):
    model.eval()  # Set the model to evaluation mode (dropout/BatchNorm behavior)
    loss_sum = correct = count = 0
    labels, probabilities = [], []
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss_sum += criterion(logits, y).item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            count += len(y)
            labels.extend(y.cpu().tolist())
            probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())
    if count == 0:
        raise ValueError('Cannot evaluate an empty loader.')
    return {'loss': loss_sum / count, 'accuracy': correct / count,
            'labels': np.asarray(labels), 'probabilities': np.asarray(probabilities)}

def validate():
    metrics = evaluate(val_loader)
    print(f"Validation - Loss: {metrics['loss']:.4f} Accuracy: {metrics['accuracy']:.4f}")
    return metrics

In [ ]:
epochs = 7  # Starting budget; choose settings using validation data only.
best_loss = float('inf')
best_state = None
history = []
for epoch in range(epochs):
    print(f'Epoch: {epoch + 1}/{epochs}')
    train_loss, train_accuracy = train()
    metrics = validate()
    history.append({'train_loss': train_loss, 'train_accuracy': train_accuracy,
                    'val_loss': metrics['loss'], 'val_accuracy': metrics['accuracy']})
    if metrics['loss'] < best_loss:
        best_loss = metrics['loss']
        # Clone tensors: state_dict alone would track later training updates.
        best_state = {key: value.detach().cpu().clone()
                      for key, value in model.state_dict().items()}
        best_epoch = epoch + 1
if best_state is None:
    raise RuntimeError('No finite validation-loss checkpoint was produced.')
model.load_state_dict(best_state)
print(f'Restored epoch {best_epoch} (validation loss {best_loss:.4f}).')

## Final test evaluation
Run after training and model choices are complete. The best validation checkpoint is restored above. The old notebook used these test images every epoch, so a strictly fresh estimate requires an external patient-disjoint holdout. Keep the default argmax decision rule; any threshold tuning belongs on validation data.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, average_precision_score)
import matplotlib.pyplot as plt

metrics = evaluate(test_loader)  # Only final evaluation uses test_loader.
y_true = metrics['labels']
scores = metrics['probabilities']  # P(PNEUMONIA)
y_pred = (scores > 0.5).astype(int)  # Matches argmax: exact ties choose NORMAL.
print(f"Test loss: {metrics['loss']:.4f}; accuracy: {metrics['accuracy']:.4f}")
print(classification_report(y_true, y_pred, labels=[0, 1],
      target_names=test_dataset.classes, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print('Pneumonia sensitivity:', tp / (tp + fn) if tp + fn else float('nan'))
print('Normal specificity:', tn / (tn + fp) if tn + fp else float('nan'))
if len(np.unique(y_true)) == 2:
    print('ROC AUC:', roc_auc_score(y_true, scores))
    print('Average precision:', average_precision_score(y_true, scores))
ConfusionMatrixDisplay(cm, display_labels=test_dataset.classes).plot(cmap='Blues')
plt.show()
plt.plot([row['train_loss'] for row in history], label='Training')
plt.plot([row['val_loss'] for row in history], label='Validation')
plt.xlabel('Epoch (zero-based)')
plt.ylabel('Cross-entropy loss')
plt.legend()
plt.show()

## Review notes and next steps
- The provided validation folder has only 16 images in the saved run. One error changes accuracy by 6.25 percentage points. Keep results exploratory; build a larger split using verified patient IDs before extensive tuning.
- Identical-image checks are a partial safeguard. They do not prove that patients are independent across splits. Do not infer patient identity from arbitrary filename fragments; obtain reliable metadata and group all images from a patient together.
- The old crop scale `(0.3, 6)` allowed areas exceeding the original image and could trigger fallback cropping. Small crops can also remove relevant anatomy. The new baseline avoids random cropping and flipping; visually inspect augmentations before using more.
- The old first dense layer had 39,322,112 parameters. Adaptive pooling reduces model size substantially while keeping the original three convolution blocks for learning.
- Training/evaluation modes, raw logits with cross-entropy, and sample-weighted batch loss were already correct.
- Compare a pretrained torchvision model later using its documented input preprocessing; do not reuse single-channel normalization blindly.
- Dataset performance alone does not establish clinical usefulness or performance on another hospital/population.

References: [torchvision crop semantics](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.RandomResizedCrop.html), [PyTorch checkpoint copying](https://docs.pytorch.org/tutorials/beginner/saving_loading_models).

## Original comments and learning notes
Original changed cells are preserved below for reference; the executable workflow above contains the corrections.

<details>
<summary>Original cell 3</summary>

```python
#@title Libraries
import numpy as np
import pandas as pd
import os

# Torch Dependencies
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as transforms # Used to apply transformations to the dataset
import torchvision.transforms.functional as F
```

</details>

<details>
<summary>Original cell 7</summary>

```python
#@title Accessing the chest_xray folder
file_list = os.listdir(path)
print(file_list)

chest_xray_path = os.path.join(path, 'chest_xray')
print(f"Contents of chest_xray: {os.listdir(chest_xray_path)}")
```

</details>

<details>
<summary>Original cell 8</summary>

```python
#@title Transform Function

width = 256
height = 256


transformation = transforms.Compose([
    # Apply RandomResizedCrop first to handle both cropping and initial resizing
    transforms.RandomResizedCrop((width, height), scale=(0.3, 6), ratio=(1,1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.5), # Operates on 3 channels
    transforms.Grayscale(num_output_channels=1), # Convert to 1 channel (grayscale)
    transforms.ToTensor(), # Converting the images to PyTorch tensors
    # Update Normalize for single channel grayscale images
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Transformation for validation and test sets (no random augmentations)
val_test_transformation = transforms.Compose([
    transforms.Resize((width, height)), # Resize to desired dimensions
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(), # Convert to PyTorch tensors
    transforms.Normalize(mean=[0.5], std=[0.5]) # Normalize with the same values as training
])

"""
It's generally best practice to apply all geometric and color augmentations first,
then convert the image to a PyTorch tensor, and finally normalize it.
"""
```

</details>

<details>
<summary>Original cell 9</summary>

```python
#@title Creating ImageFolder & DataLoaders

# Define the paths for train, validation, and test directories
train_dir = os.path.join(chest_xray_path, 'train')
val_dir = os.path.join(chest_xray_path, 'val')
test_dir = os.path.join(chest_xray_path, 'test')

# Create ImageFolder datasets
train_dataset = torchvision.datasets.ImageFolder(root=train_dir, transform=transformation)
val_dataset = torchvision.datasets.ImageFolder(root=val_dir, transform=val_test_transformation)
test_dataset = torchvision.datasets.ImageFolder(root=test_dir, transform=val_test_transformation)

# Define batch sizes
batch_size = 32

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

valid_N = len(val_loader.dataset)

# Print the number of samples in each dataset
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

print(f"Classes: {train_dataset.classes}")
```

</details>

<details>
<summary>Original cell 11</summary>

```python
n_classes = 2
kernel_size = 3 # Size of the convolutional filter
img_chs = 1 # 1 because the x-ray images are grayscale
flattened_img_size = 75 * 32 * 32 # Corrected based on 256x256 input after 3 MaxPool2d layers

model = nn.Sequential(
    # First convolution
    nn.Conv2d(img_chs, 25, kernel_size, stride=1, padding=1),  # Output 25 channels, spatial size 256x256
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # Downsamples the feature map to 128x128
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # Output 50 channels, spatial size 128x128
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # Output 50 channels, spatial size 64x64
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # Output 75 channels, spatial size 64x64
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # Output 75 channels, spatial size 32x32
    # Flatten to Dense
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.6),
    nn.ReLU(),
    nn.Linear(512, n_classes)
)

# batch normalization scales the values in the hidden layers to improve training.
# The padding helps maintain the spatial dimensions.
```

</details>

<details>
<summary>Original cell 12</summary>

```python
#@title Compile the model
model = torch.compile(model.to(device))
model
```

</details>

<details>
<summary>Original cell 13</summary>

```python
#@title Criterion and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)
```

</details>

<details>
<summary>Original cell 16</summary>

```python
#@title Training Function

def train():
  total_loss = 0
  total_correct = 0
  total_samples = 0

  model.train()
  for x,y in train_loader:
      x, y = x.to(device), y.to(device) # Move data to the specified device
      output= model(x)
      optimizer.zero_grad()
      batch_loss = criterion(output, y)
      batch_loss.backward()
      optimizer.step()

      total_loss += batch_loss.item() * len(x) # Accumulate weighted loss
      pred = output.argmax(dim=1, keepdim=True)
      total_correct += pred.eq(y.view_as(pred)).sum().item()
      total_samples += len(x)

  avg_loss = total_loss / total_samples
  avg_accuracy = total_correct / total_samples
  print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(avg_loss, avg_accuracy))
```

</details>

<details>
<summary>Original cell 17</summary>

```python
#@title Validation Function
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

def validate():
    model.eval() # Set the model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in test_loader: # Using test_loader for final evaluation
            x, y = x.to(device), y.to(device)
            output = model(x)
            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Calculate Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:")
    print(cm)

    # Display Confusion Matrix visually
    plt.figure(figsize=(4, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_dataset.classes, yticklabels=test_dataset.classes)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

    # Generate and print classification report
    report = classification_report(all_labels, all_preds, target_names=test_dataset.classes)
    print("\nClassification Report:")
    print(report)
```

</details>

<details>
<summary>Original cell 18</summary>

```python
epochs = 7

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()
```

</details>